# 18. Named Entity Recognition

## Overview
Named Entity Recognition (NER) identifies spans of text that refer to entities such as people, organizations, locations, dates, and monetary values.

This notebook demonstrates NLTK's named-entity chunker, shows how to inspect the resulting tree, and converts recognized entities into a structured list.

## 1. Setup

The NLTK NER pipeline requires tokenization, POS tagging, and a pretrained named-entity chunker. Download the resources below if they are not already installed.

In [1]:
import nltk

nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("maxent_ne_chunker_tab")
nltk.download("words")

[nltk_data] Downloading package punkt_tab to /home/mg/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/mg/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /home/mg/nltk_data...
[nltk_data]   Package maxent_ne_chunker_tab is already up-to-date!
[nltk_data] Downloading package words to /home/mg/nltk_data...
[nltk_data]   Package words is already up-to-date!


True

## 2. Recognize entities in a sentence

`ne_chunk` expects POS-tagged tokens. With `binary=False`, recognized entity labels can include categories such as `PERSON`, `ORGANIZATION`, and `GPE` (geopolitical entity).

In [2]:
from nltk import word_tokenize, pos_tag, ne_chunk

text = (
    "Microsoft announced a new research partnership with the University of Oxford. "
    "Satya Nadella discussed the project in London on Tuesday."
)

tokens = word_tokenize(text)
tagged_tokens = pos_tag(tokens)
ner_tree = ne_chunk(tagged_tokens, binary=False)

print(ner_tree)

(S
  (PERSON Microsoft/NNP)
  announced/VBD
  a/DT
  new/JJ
  research/NN
  partnership/NN
  with/IN
  the/DT
  (ORGANIZATION University/NNP)
  of/IN
  (GPE Oxford/NNP)
  ./.
  (PERSON Satya/NNP Nadella/NNP)
  discussed/VBD
  the/DT
  project/NN
  in/IN
  (GPE London/NNP)
  on/IN
  Tuesday/NNP
  ./.)


## 3. Visualize the entity tree

In notebook environments, `draw()` may open a separate window and depends on the local graphical setup. The tree's textual representation works in ordinary notebook sessions.

In [3]:
print(ner_tree.pformat(margin=100))

(S
  (PERSON Microsoft/NNP)
  announced/VBD
  a/DT
  new/JJ
  research/NN
  partnership/NN
  with/IN
  the/DT
  (ORGANIZATION University/NNP)
  of/IN
  (GPE Oxford/NNP)
  ./.
  (PERSON Satya/NNP Nadella/NNP)
  discussed/VBD
  the/DT
  project/NN
  in/IN
  (GPE London/NNP)
  on/IN
  Tuesday/NNP
  ./.)


## 4. Extract entity spans

The following helper collects each named entity as a label and a text span. Multi-token entities are joined into a single string.

In [4]:
def extract_named_entities(chunk_tree):
    entities = []

    for node in chunk_tree:
        if hasattr(node, "label"):
            entity_text = " ".join(token for token, _ in node.leaves())
            entities.append({
                "text": entity_text,
                "label": node.label(),
            })

    return entities

entities = extract_named_entities(ner_tree)
for entity in entities:
    print(entity)

{'text': 'Microsoft', 'label': 'PERSON'}
{'text': 'University', 'label': 'ORGANIZATION'}
{'text': 'Oxford', 'label': 'GPE'}
{'text': 'Satya Nadella', 'label': 'PERSON'}
{'text': 'London', 'label': 'GPE'}


## 5. Store the entities in a DataFrame

Structured output is easier to filter, sort, and export than a tree representation.

In [5]:
import pandas as pd

entities_df = pd.DataFrame(entities, columns=["text", "label"])
display(entities_df)

,text,label
0,Microsoft,PERSON
1,University,ORGANIZATION
2,Oxford,GPE
3,Satya Nadella,PERSON
4,London,GPE


## 6. Count entity types

Entity-type counts can provide a quick summary of the entities found in a document.

In [6]:
if not entities_df.empty:
    display(
        entities_df["label"]
        .value_counts()
        .rename_axis("entity_type")
        .reset_index(name="count")
    )
else:
    print("No named entities were recognized in this text.")

,entity_type,count
0,PERSON,2
1,GPE,2
2,ORGANIZATION,1


## 7. Process multiple sentences

This helper runs the same pipeline over a full text and returns a flat list of entities with their sentence context.

In [7]:
def extract_entities_from_text(text):
    all_entities = []

    for sentence in nltk.sent_tokenize(text):
        sentence_tokens = word_tokenize(sentence)
        sentence_tags = pos_tag(sentence_tokens)
        sentence_tree = ne_chunk(sentence_tags, binary=False)

        for entity in extract_named_entities(sentence_tree):
            all_entities.append({
                "sentence": sentence,
                "text": entity["text"],
                "label": entity["label"],
            })

    return pd.DataFrame(all_entities, columns=["sentence", "text", "label"])

document = (
    "Apple opened a new office in Berlin. "
    "Angela Merkel previously served as chancellor of Germany. "
    "The company said the office will support European customers."
)

document_entities = extract_entities_from_text(document)
display(document_entities)

,sentence,text,label
0,Apple opened a new office in Berlin.,Apple,PERSON
1,Apple opened a new office in Berlin.,Berlin,GPE
2,Angela Merkel previously served as chancellor ...,Angela,PERSON
3,Angela Merkel previously served as chancellor ...,Merkel,PERSON
4,Angela Merkel previously served as chancellor ...,Germany,GPE
5,The company said the office will support Europ...,European,GPE


## Interpretation and limitations

- NER identifies entity mentions; it does not automatically resolve which real-world entity a mention refers to.
- The NLTK chunker is a general-purpose English baseline and may miss entities or assign incorrect labels.
- Results can vary with capitalization, context, domain, and the model's training data.
- For production systems, evaluate on representative annotated data and consider modern transformer-based NER models.

## Summary

NER converts unstructured text into entity spans and categories. This notebook shows how to run NLTK's pipeline, extract multi-word entities, and organize the results for downstream analysis.